In [ ]:
import numpy as np
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, TransformerMixin
from category_encoders import TargetEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.compose import TransformedTargetRegressor
from sklearn.impute import KNNImputer
from sklearn.linear_model import Lasso, Ridge
from sklearn.linear_model import ElasticNet

In [ ]:
preprocess_df_test = pd.read_csv("preprocess_df.csv")# ??????

In [ ]:
#-------------------------------Price_Model-------------------------------------------------------------
x = preprocess_df_test.drop('Price', axis=1)
y = preprocess_df_test['Price'] 

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=111
)
                     
log_cols = ['area', 'gas_fee'] 

log1p_cols = ['density_ratio', 'reflect_sentiment', 'park_area', 'greening', 'plot_ratio','property_fee']

knn_num_features = ['built_year','transaction_year', 'elevator_ratio']

house_allo_cols = ['room', 'hall', 'kitchen', 'bathroom']

binary_features = ['east', 'south', 'west', 'north', 'ownership']

low_cat_features = ['city', 'build_struc', 'decoration', 'use_of_house', 'tran_right', 'floor', 'wat_sup', 'ele_sup']

high_cat_district = ['district']  #target encoding
high_cat_plate = ['plate']  #clustering

class Winsorizer(BaseEstimator, TransformerMixin): #BE????? TM????pipeline??
    def __init__(self, lower_q=0.01, upper_q=0.99):
        self.lower_q = lower_q
        self.upper_q = upper_q
        self.low_ = None
        self.high_ = None

    def fit(self, X, y=None):
        X = np.asarray(X)
        self.low_ = np.nanpercentile(X, 100 * self.lower_q, axis=0)
        self.high_ = np.nanpercentile(X, 100 * self.upper_q, axis=0)
        return self

    def transform(self, X):
        X = np.asarray(X).astype(float)
        return np.clip(X, self.low_, self.high_)

class LonLatClusterTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=8):
        self.n_clusters = n_clusters
        self.kmeans = None
        
    def fit(self, X, y=None):
        lon_lat = X[['lon', 'lat']].values #Kmeans??????
        self.kmeans = KMeans(n_clusters=self.n_clusters, random_state=42)
        self.kmeans.fit(lon_lat)
        return self
        
    def transform(self, X):
        lon_lat = X[['lon', 'lat']].values
        clusters = self.kmeans.predict(lon_lat)
        return clusters.reshape(-1, 1) #???????????(label)
        
class PlateClusterTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10):
        self.n_clusters = n_clusters
        self.kmeans = None
        self.encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False) #??np??
        
    def fit(self, X, y=None):
        plate_data = X.values.reshape(-1, 1)
        plate_encoded = self.encoder.fit_transform(plate_data)
        self.kmeans = KMeans(n_clusters=self.n_clusters, random_state=42)
        self.kmeans.fit(plate_encoded)
        return self
        
    def transform(self, X):
        plate_data = X.values.reshape(-1, 1)
        plate_encoded = self.encoder.transform(plate_data)
        clusters = self.kmeans.predict(plate_encoded)
        return clusters.reshape(-1, 1) #sklearn Transformer ??????

knn_num_pipeline = Pipeline([
    ('ref_imputer', SimpleImputer(strategy='median')),
    ('knn_imputer', KNNImputer(n_neighbors=5)), 
    ('winsor', Winsorizer(lower_q=0.001, upper_q=0.999)), 
    ('scaler', StandardScaler()) 
])

log_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(np.log)), 
    ('winsor', Winsorizer(lower_q=0.001, upper_q=0.999)),
    ('scaler', StandardScaler())
])

log1p_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log1p', FunctionTransformer(np.log1p)),
    ('winsor', Winsorizer(lower_q=0.001, upper_q=0.999)),
    ('scaler', StandardScaler())
])

house_allo_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('winsor', Winsorizer(lower_q=0.001, upper_q=0.999)),
    ('scaler', StandardScaler())
])

low_cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

binary_pipeline = Pipeline([
    ('passthrough', FunctionTransformer(lambda x: x))  # ????
])


preprocessor = ColumnTransformer([
    ('geo_cluster', LonLatClusterTransformer(n_clusters=40), ['lon', 'lat']),
    ('log_cols', log_pipeline, log_cols),
    ('log1p_num', log1p_pipeline, log1p_cols),
    ('built_year_knn_impute', knn_num_pipeline, knn_num_features),
    ('house_allo', house_allo_pipeline, house_allo_cols),
    ('binary', binary_pipeline, binary_features),
    ('low_cat', low_cat_pipeline, low_cat_features),
    ('district_target', TargetEncoder(cols=['district'], smoothing=50), ['district']),
    ('plate_cluster', PlateClusterTransformer(n_clusters=10), ['plate'])
], remainder='drop')

base_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor',  Ridge(alpha=200)) #LinearRegression(), Lasso(), Ridge(), ElasticNet()
])

ttr = TransformedTargetRegressor(regressor=base_pipeline,
                                     func=np.log,
                                     inverse_func=np.exp)

ttr.fit(x_train, y_train)

y_pred = ttr.predict(x_test)
y_pred_in = ttr.predict(x_train)

mae = mean_absolute_error(y_test, y_pred)
mae_in = mean_absolute_error(y_train, y_pred_in)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae}")
print(f"MAEIN: {mae_in}")
print(f"MSE: {mse}")
print(f"R?: {r2}")

In [ ]:
from sklearn.model_selection import cross_val_score

cv_folds = 6 
scoring_metric = 'neg_mean_absolute_error'

cv_scores = cross_val_score(
    estimator=ttr,
    X=x_train,                
    y=y_train,                 
    cv=cv_folds,              
    scoring=scoring_metric 
)

print(f"cv_avg: {cv_scores.mean()}")

In [ ]:
#-------------------------------process_rent-------------------------------------------------------------
def floor2num(floor_str):
    if pd.isna(floor_str):
        return np.nan
    floor_str = str(floor_str).strip()

    if "?" in floor_str:
        return 3
    elif "?" in floor_str:
        return 2
    elif "?" in floor_str:
        return 1

    if floor_str[0].isdigit():
        frac = eval(floor_str)
        if frac < 0.3:
            return 1
        elif frac > 0.6:
            return 3
        else:
            return 2

def apply_floor_conversion(df, col_name="??"):
    df_copy = df.copy()
    df_copy["floor"] = df_copy[col_name].apply(floor2num)
    return df_copy          

def split_house_type(df, col_name="??"):
    df_copy = df.copy()
    patterns = {
        'room': r'(\d+)?',    
        'hall': r'(\d+)?',    
        'kitchen': r'(\d+)?', 
        'bathroom': r'(\d+)?' 
    }
    for new_col, pattern in patterns.items():
        df_copy[new_col] = df_copy[col_name].apply(
            lambda x: int(re.search(pattern, str(x)).group(1)) #??????????????(\d+)
            if pd.notnull(x) and re.search(pattern, str(x))
            else np.nan
        )
    return df_copy

def remove_string(df, col_name, strings_to_remove):
    df_copy = df.copy()
    pattern = '|'.join(re.escape(s) for s in strings_to_remove) #escape()???????
    df_copy[col_name] = df_copy[col_name].str.replace(pattern, '', regex=True)
    return df_copy

def change_toward(df, col_name="??"):
    df_copy = df.copy()
    directions = {
        'east': '?',   
        'south': '?',  
        'west': '?',   
        'north': '?'  
    }
    for new_col, direction_char in directions.items():
        df_copy[new_col] = df_copy[col_name].apply(
            lambda x: 1 if pd.notna(x) and direction_char in str(x) else 0
        )
    return df_copy

def water2num(water_str):
    if "??" in str(water_str):
        return 2
    elif "??" in str(water_str):
        return 1
    else:
        return np.nan

def apply_water_conversion(df, col_name="??"):
    df_copy = df.copy()
    df_copy[col_name] = df_copy[col_name].apply(water2num) 
    return df_copy

def ele2num(ele_str):
    if "??" in str(ele_str):
        return 2
    elif "??" in str(ele_str):
        return 1
    else:
        return np.nan

def apply_ele_conversion(df, col_name="??"):
    df_copy = df.copy()
    df_copy[col_name] = df_copy[col_name].apply(ele2num) 
    return df_copy

def yesno2num(w_str):
    if "?" in str(w_str):
        return 1
    elif "?" in str(w_str):
        return 0
    else:
        return np.nan

def apply_yesno2num(df, col_name=None):
    df_copy = df.copy()
    df_copy[col_name] = df_copy[col_name].apply(yesno2num) 
    return df_copy

def rent2num(r_str):
    if "??" in str(r_str):
        return 2
    elif "??" in str(r_str):
        return 1
    else:
        return np.nan

def apply_rent2num(df, col_name="????"):
    df_copy = df.copy()
    df_copy[col_name] = df_copy[col_name].apply(rent2num) 
    return df_copy

def avgdense_ratio(df, col1="????", col2="????"):
    df_copy = df.copy()
    
    new_col_name = "density_ratio" 
    safe_col1 = df_copy[col1].replace(0, np.nan)
    safe_col2 = df_copy[col2].replace(0, np.nan)
    df_copy[new_col_name] = safe_col1 / safe_col2
    
    return df_copy

def cal_deal_time(df, col_name="????", base_year=2025):
    df_copy = df.copy()
    
    df_copy["transaction_year"] = df_copy[col_name].apply( 
        lambda x: base_year - int(str(x)[:4]) if pd.notna(x) and str(x).strip() else np.nan
    ) 
    return df_copy

facility = ["?", "??", "??", "???", "???"]
def check_facility(df, col_name):
    df_copy = df.copy()
    
    for f in facility:
        df_copy[f] = np.nan 
    
    def process_row(row_idx, s):
        if pd.isna(s):
            s = ""
        else:
            s = str(s).strip()
        
        s_facilities = [item.strip() for item in s.split("?") if item.strip()]
        
        for f in facility:
            if f in s_facilities:
                df_copy.at[row_idx, f] = 1
            else:
                df_copy.at[row_idx, f] = 0
    
    for row_idx, s in df[col_name].items():
        process_row(row_idx, s)
    
    return df_copy

build_cat = ["????", "??", "??", "??", "??", "??"]
def check_build_cat(df, col_name):
    df_copy = df.copy()
    
    for bc in build_cat:
        df_copy[bc] = np.nan 
    
    def process_row(row_idx, s):
        if pd.isna(s):
            s = ""
        else:
            s = str(s).strip()
        
        s_bc = [item.strip() for item in s.split("/") if item.strip()]
        
        for bc in build_cat:
            if bc in s_bc:
                df_copy.at[row_idx, bc] = 1
            else:
                df_copy.at[row_idx, bc] = 0
    
    for row_idx, s in df[col_name].items():
        process_row(row_idx, s)
    
    return df_copy

def take_year(df, col_name="????"):
    df_copy = df.copy()

    def process_single_value(s):
        try:
            year_pos = s.find("?")
            if year_pos >= 4:
                year_str = s[year_pos - 4 : year_pos]
                return int(year_str)
            else:
                return None

        except:
            return None
    
    df_copy[col_name] = df_copy[col_name].apply(process_single_value)
    return df_copy

def interval_covert(df, col_name=None):
    df_copy = df.copy()

    def count_interval(s):
        
        s = str(s).strip()
        
        try:
            if "-" in str(s):
                dash_pos = s.find("-")
                a = s[:dash_pos].strip()
                b = s[dash_pos + 1:].strip()
                i_mean = (float(b)+float(a))/2
                return i_mean
                
            else:
                return float(s)
        except:
            return None

    df_copy[col_name] = df_copy[col_name].apply(count_interval)
    return df_copy


def preprocess_before_pipeline(df, config=None): #???config??????
    if config is None:
        config = {}

    floor_col = config.get('floor_col', '??')
    house_type_col = config.get('house_type_col', '??')
    toward_col = config.get('toward_col', '??')
    house_area_col = config.get('house_area_col', '??')
    elev_col = config.get('elev_col', '??')
    deal_time_col = config.get('deal_time_col', '????')
    way2rent_col = config.get('way2rent_col', '????')
    house_total_col = config.get('house_total_col', '????')
    building_total_col = config.get('building_total_col', '????')
    gas_fee_col = config.get('gas_fee_col', '??')
    wat_sup_col = config.get('wat_sup_col', '??')
    ele_sup_col = config.get('ele_sup_col', '??')
    
    facility_col = config.get('facility_col', '????')
    build_cat_col = config.get('build_cat_col', '????')
    built_year_col = config.get('built_year_col', '????')
    greening_rate_col = config.get('greening_rate_col', '? ? ?')
    property_fee_col = config.get('property_fee_col', '? ? ?')
    gas_feenum_col = config.get('gas_fee_col', '???')
    
    
    df_processed = df.copy()
    df_processed = remove_string(df_processed, floor_col, "?")
    df_processed = apply_floor_conversion(df_processed, floor_col)
    
    df_processed = split_house_type(df_processed, house_type_col)
    
    df_processed = remove_string(df_processed, house_area_col, "?")
    df_processed[house_area_col] = pd.to_numeric(df_processed[house_area_col], errors='coerce')
    
    df_processed = change_toward(df_processed, toward_col)

    df_processed = cal_deal_time(df_processed, deal_time_col)
    
    #df_processed = apply_rent2num(df_processed, way2rent_col)

    df_processed = apply_yesno2num(df_processed, elev_col)
    
    df_processed = apply_yesno2num(df_processed, gas_fee_col)

    df_processed = apply_water_conversion(df_processed, wat_sup_col)
    df_processed = apply_ele_conversion(df_processed, ele_sup_col)
    
    df_processed = check_facility(df_processed, facility_col)
    df_processed = check_build_cat(df_processed, build_cat_col)
    df_processed = take_year(df_processed, built_year_col)
    
    df_processed = remove_string(df_processed, greening_rate_col, "%")
    df_processed[greening_rate_col] = pd.to_numeric(df_processed[greening_rate_col], errors='coerce')
    df_processed = remove_string(df_processed, property_fee_col, "?/?/?")
    df_processed = interval_covert(df_processed, property_fee_col)
    df_processed[property_fee_col] = pd.to_numeric(df_processed[property_fee_col], errors='coerce')
    df_processed = remove_string(df_processed, gas_feenum_col, "?/m?")
    df_processed =interval_covert(df_processed, gas_feenum_col)
    df_processed[gas_feenum_col] = pd.to_numeric(df_processed[gas_feenum_col], errors='coerce')
    
    df_processed = remove_string(df_processed, house_total_col, "?")
    df_processed = remove_string(df_processed, building_total_col, "?")
    df_processed[house_total_col] = pd.to_numeric(df_processed[house_total_col], errors='coerce')
    df_processed[building_total_col] = pd.to_numeric(df_processed[building_total_col], errors='coerce')
    df_processed = avgdense_ratio(df_processed, house_total_col, building_total_col)
    
    columns_to_drop = [floor_col, house_type_col, toward_col, deal_time_col]
    existing_cols_to_drop = [col for col in columns_to_drop if col in df_processed.columns]
    df_processed = df_processed.drop(columns=existing_cols_to_drop)
    
    return df_processed

column_mapping = {
    "??": "city", 
    "??": "district", 
    "??": "plate", 
    "??": "area",
    "??": "year",
    "????": "way2rent",
    "??": "elevator",
    "??": "wat_sup",
    "??": "ele_sup",
    "??": "gas",
    "?": "bed", 
    "??": "wardrobe", 
    "??": "air_condi", 
    "???": "wash_mach",
    "???": "water_heat",
    "????": "dwelling",
    "??": "ground_comm",
    "??": "commerce",
    "??": "carport",
    "??": "apart",
    "??": "villa",
    "??": "period2rent",
    "????": "built_year",
    "? ? ?": "greening_rate",
    "? ? ?": "plot_ratio",
    "? ? ?": "property_fee",
    "????": "build_struc",
    "???": "gas_feenum",
    "???": "park_area"
}

def fraction_to_float(frac_str):
    if pd.isna(frac_str):
        return None
    
    try:
        parts = str(frac_str).split('/')
        if len(parts) == 2:
            numerator = float(parts[0])
            denominator = float(parts[1])
            return numerator / denominator
        else:
            return float(frac_str)
    except:
        return None

In [ ]:
test_rent = pd.read_csv("ruc_Class25Q2_train_rent.csv") #????
test_rent_senti = pd.read_csv("sentiment_rent.csv")
test_rent_sentiment = test_rent_senti.copy()
test_rent["reflect_sentiment"] = test_rent_senti["reflect_sentiment"]
test_rent_sentiment = test_rent.copy()
preprocess_before_df = preprocess_before_pipeline(test_rent_sentiment, config=None)
preprocess_before_df.drop(columns=["????", "coord_x", "coord_y", "????", "????", "????",
                                  "????", "????", "????", "???", "????", "????",
                                  "??", "??", "????"], inplace=True)
preprocess_before_df.columns = [column_mapping.get(col, col) for col in preprocess_before_df.columns]
preprocess_before_df.info()
preprocess_df_test = preprocess_before_df.copy()

In [ ]:
#-------------------------------Rent_Model-------------------------------------------------------------
x = preprocess_df_test.drop('Price', axis=1)
y = preprocess_df_test['Price'] 

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=111
)

log_cols = ['gas_feenum', 'park_area']

log1p_cols = ['density_ratio', 'reflect_sentiment', 'plot_ratio', 'greening_rate', 'property_fee']

num_cols = ['area']

knn_num_features = ['built_year','transaction_year', 'year'] #knn

house_allo_cols = ['room', 'hall', 'bathroom', 'dwelling', 'ground_comm', 'commerce', 'carport', 'apart', 'villa']#knn

binary_features = ['east', 'south', 'west', 'north', 'bed', 'wardrobe', 'air_condi', 'wash_mach', 'water_heat']

low_cat_features = ['city', 'way2rent', 'floor', 'wat_sup', 'ele_sup', 'build_struc']

high_cat_district = ['district']  #target encoding
high_cat_period2rent = ['period2rent']
high_cat_plate = ['plate']  #clustering

class Winsorizer(BaseEstimator, TransformerMixin): #BE????? TM????pipeline??
    def __init__(self, lower_q=0.01, upper_q=0.99):
        self.lower_q = lower_q
        self.upper_q = upper_q
        self.low_ = None
        self.high_ = None

    def fit(self, X, y=None):
        X = np.asarray(X)
        self.low_ = np.nanpercentile(X, 100 * self.lower_q, axis=0)
        self.high_ = np.nanpercentile(X, 100 * self.upper_q, axis=0)
        return self

    def transform(self, X):
        X = np.asarray(X).astype(float)
        return np.clip(X, self.low_, self.high_)

class LonLatClusterTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=8):
        self.n_clusters = n_clusters
        self.kmeans = None
        
    def fit(self, X, y=None):
        lon_lat = X[['lon', 'lat']].values #Kmeans??????
        self.kmeans = KMeans(n_clusters=self.n_clusters, random_state=42)
        self.kmeans.fit(lon_lat)
        return self
        
    def transform(self, X):
        lon_lat = X[['lon', 'lat']].values
        clusters = self.kmeans.predict(lon_lat)
        return clusters.reshape(-1, 1) #???????????(label)
        
class PlateClusterTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10):
        self.n_clusters = n_clusters
        self.kmeans = None
        self.encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False) #??np??
        
    def fit(self, X, y=None):
        plate_data = X.values.reshape(-1, 1)
        plate_encoded = self.encoder.fit_transform(plate_data)
        self.kmeans = KMeans(n_clusters=self.n_clusters, random_state=42)
        self.kmeans.fit(plate_encoded)
        return self
        
    def transform(self, X):
        plate_data = X.values.reshape(-1, 1)
        plate_encoded = self.encoder.transform(plate_data)
        clusters = self.kmeans.predict(plate_encoded)
        return clusters.reshape(-1, 1) #sklearn Transformer ??????

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('winsor', Winsorizer(lower_q=0.01, upper_q=0.99)), 
    ('scaler', StandardScaler()) 
])

log_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(np.log)), 
    ('winsor', Winsorizer(lower_q=0.01, upper_q=0.99)),
    ('scaler', StandardScaler())
])

log1p_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log1p', FunctionTransformer(np.log1p)),
    ('winsor', Winsorizer(lower_q=0.01, upper_q=0.99)),
    ('scaler', StandardScaler())
])

knn_num_pipeline = Pipeline([
    ('ref_imputer', SimpleImputer(strategy='median')),
    ('knn_imputer', KNNImputer(n_neighbors=5)), 
    ('winsor', Winsorizer(lower_q=0.01, upper_q=0.99)), 
    ('scaler', StandardScaler()) 
])

house_allo_pipeline = Pipeline([
    ('ref_imputer', SimpleImputer(strategy='median')),
    ('knn_imputer', KNNImputer(n_neighbors=5)),
    ('winsor', Winsorizer(lower_q=0.01, upper_q=0.99)),
    ('scaler', StandardScaler())
])

low_cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

binary_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
])


preprocessor = ColumnTransformer([
    ('geo_cluster', LonLatClusterTransformer(n_clusters=25), ['lon', 'lat']),
    ('log_cols', log_pipeline, log_cols),
    ('log1p_num', log1p_pipeline, log1p_cols),
    ('num_feature', num_pipeline, num_cols),
    ('knn_num_impute', knn_num_pipeline, knn_num_features),
    ('house_allo', house_allo_pipeline, house_allo_cols),
    ('binary', binary_pipeline, binary_features),
    ('low_cat', low_cat_pipeline, low_cat_features),
    ('district_target', TargetEncoder(cols=['district'], smoothing=15), ['district']),
    ('period2rent', TargetEncoder('period2rent', smoothing=15), ['period2rent']),
    ('plate_cluster', PlateClusterTransformer(n_clusters=10), ['plate'])
], remainder='drop')

base_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression()) #LinearRegression()
])

ttr = TransformedTargetRegressor(regressor=base_pipeline,
                                     func=np.log,
                                     inverse_func=np.exp)

ttr.fit(x_train, y_train)

y_pred = ttr.predict(x_test)
y_pred_in = ttr.predict(x_train)

mae = mean_absolute_error(y_test, y_pred)
mae_in = mean_absolute_error(y_train, y_pred_in)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae}")
print(f"MAEIN: {mae_in}")
print(f"MSE: {mse}")
print(f"R?: {r2}")

In [ ]:
from sklearn.model_selection import cross_val_score

cv_folds = 6 
scoring_metric = 'neg_mean_absolute_error'

cv_scores = cross_val_score(
    estimator=ttr,
    X=x_train,                
    y=y_train,                 
    cv=cv_folds,              
    scoring=scoring_metric 
)

print(f"cv_avg: {cv_scores.mean()}")

In [ ]:
print(ttr.regressor)